In [0]:
%run "./School-Setup"

In [0]:
%python
stream_df = spark.readStream.table("workspace.default.courses_csv")

In [0]:
%python
stream_df.createOrReplaceTempView("courses_streaming_tmp_vw")

In [0]:
%fs ls 'dbfs:/Workspace/Shared/DEA/checkpoints/'

In [0]:
%sql
SELECT * FROM courses_streaming_tmp_vw

In [0]:
%python
from pyspark.sql.functions import count

# Query the streaming view with aggregation
result_df = spark.sql("""
  SELECT instructor, count(course_id) AS total_courses
  FROM courses_streaming_tmp_vw
  GROUP BY instructor
""")

# Write to memory with explicit checkpoint
result_df.writeStream \
    .format("memory") \
    .queryName("instructor_counts_view") \
    .outputMode("complete") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "/Workspace/Users/adyasejo@gmail.com/Demo/checkpoints/instructor_counts_1787668650") \
    .start()

In [0]:
CREATE OR REPLACE TEMP VIEW instructor_counts_tmp_vw AS (
 SELECT instructor, count(course_id) AS total_courses
 FROM courses_streaming_tmp_vw
 GROUP BY instructor
)

In [0]:
use catalog workspace

In [0]:
%python
result_stream_df = spark.table("instructor_counts_tmp_vw")

In [0]:
%python
(result_stream_df.writeStream 
                 .trigger(processingTime='3 seconds')
                 .outputMode("complete")
                 .option("checkpointLocation", "dbfs:/Workspace/Shared/DEA/checkpoints/instructor_counts")
                 .table("instructor_counts")

In [0]:
%sql
SELECT * FROM instructor_counts

In [0]:
INSERT INTO workspace.default.courses_csv
values ("C16", "Generative AI", "Pierre B.", "Computer Science", 25, NULL),
       ("C17", "Embedded Systems", "Julia S.", "Computer Science", 30, NULL),
       ("C18", "Virtual Reality", "Bernard M.", "Computer Science", 35, NULL)